In [1]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
import pickle

2026-03-20 19:20:25.020586: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-20 19:20:25.072924: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-20 19:20:25.072982: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-20 19:20:25.074552: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-20 19:20:25.082545: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-20 19:20:26.724092: W tensorflow/compiler/tf2tensorrt/utils/py_utils.

In [3]:
#load ann model, sclar pickle file, onehot encoder pickle file
model = load_model('model_1.h5')
with open('scaler_1.pkl','rb') as f:
    scaler = pickle.load(f)

with open('one_hot_encoder_geography_1.pkl','rb') as f:
    one_hot_encoder_geography = pickle.load(f)

with open('label_encoder_gender_1.pkl','rb') as f:
    gender_label_encoder = pickle.load(f)

In [4]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [5]:
geo_encoded = one_hot_encoder_geography.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = one_hot_encoder_geography.get_feature_names_out(['Geography']))

/home/hemalatha.pallem/Desktop/genai/annclassification/venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [6]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [9]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [11]:
## Encode categorical variables
input_df['Gender']=gender_label_encoder.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [12]:
input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [13]:
input_scaled = scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [15]:
prediction = model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 48ms/step


array([[0.03308188]], dtype=float32)

In [16]:
prediction_prob = prediction[0][0]

In [17]:
if prediction_prob > 0.5:
    print(f"The customer is likely to churn with a probability of {prediction_prob:.2f}")
else:
    print(f"The customer is not likely to churn with a probability of {prediction_prob:.2f}")

The customer is not likely to churn with a probability of 0.03
